# ThreadLearn — Eval: Fine-tuned Model v2 + Full Pipeline
`anha12/threadlearn-qwen2.5-coder-1.5b-merged-v2` (QLoRA fine-tuned v2) **với** BM25 + AST + RAG prompt

30 real bugs từ production npm packages. So sánh v2+pipeline với v1+pipeline để đánh giá cải thiện.

In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub rank-bm25

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set. Add to Kaggle Secrets.")

login(token=HF_TOKEN)
print("✅ Logged in to Hugging Face")

In [ ]:
MODEL_ID = "anha12/threadlearn-qwen2.5-coder-1.5b-merged-v2"
print(f"📥 Loading {MODEL_ID} ...")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                          bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                              device_map="auto", trust_remote_code=True)
model.eval()
print("✅ Fine-tuned model v2 loaded")

In [ ]:
import re, json, os
from rank_bm25 import BM25Okapi

STOPWORDS = {
    "the","a","an","is","in","on","at","to","for","of","and","or","with",
    "this","that","it","be","are","was","were","has","have","had","do","does",
    "did","not","by","from","as","if","when","then","so","but","also","can",
    "will","use","used","using","should","would","could","may","each","how",
    "what","which","who","into","after","before","its","their","they","we",
    "you","he","she","i","me",
}

def _split_camel(token):
    p = re.sub(r"([a-z])([A-Z])", r"\1 \2", token)
    p = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", p)
    return p.lower().split()

def bm25_tokenize(text):
    tokens = []
    for raw in re.split(r"[^a-zA-Z0-9]+", text):
        if not raw: continue
        for sub in _split_camel(raw):
            if len(sub) >= 2 and sub not in STOPWORDS:
                tokens.append(sub)
    return tokens

def build_bm25_index(docs):
    corpus = [bm25_tokenize(f"{d.get('title','')} {d.get('content','')}") for d in docs]
    return BM25Okapi(corpus), docs

def bm25_search(index, docs, query, top_k=3):
    q_tokens = bm25_tokenize(query)
    if not q_tokens: return []
    scores = index.get_scores(q_tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k*5]
    results, seen = [], set()
    for idx in top_idx:
        if scores[idx] > 0:
            doc = docs[idx]
            base = re.sub(r'\s*\[.*?\]\s*$', '', doc.get('title','')).strip()
            if base not in seen:
                seen.add(base)
                d = dict(doc); d["bm25_score"] = round(float(scores[idx]), 3)
                results.append(d)
                if len(results) >= top_k: break
    return results

JS_KEYWORDS = {
    "var","let","const","function","return","if","else","for","while","do",
    "try","catch","throw","new","class","extends","import","export","from",
    "async","await","then","null","undefined","true","false","typeof","instanceof",
}

def extract_keywords(code):
    clean = re.sub(r"'[^']*'|\"[^\"]*\"", " ", code)
    clean = re.sub(r"//.*", " ", clean)
    tokens = re.findall(r"[a-zA-Z_$][a-zA-Z0-9_$]*", clean)
    kw = [t for t in tokens if t.lower() not in JS_KEYWORDS and len(t) >= 3]
    seen, result = set(), []
    for k in kw:
        if k not in seen: seen.add(k); result.append(k)
    return " ".join(result[:20])

def build_prompt(code, docs):
    if docs:
        parts = [
            f"Reference {i} — {d.get('title','')} [{d.get('category','')}]:\n{d.get('content','')[:500]}"
            for i, d in enumerate(docs, 1)
        ]
        ctx = "\n\n".join(parts)
        return f"JavaScript concurrency reference docs:\n\n{ctx}\n\n---\n\nConvert to concurrent JavaScript:\n\n{code}\n"
    return f"Convert to concurrent JavaScript:\n\n{code}\n"

# Find KB file — search all of /kaggle/input recursively
KB_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "knowledge_base" in f and f.endswith(".json"):
            KB_PATH = os.path.join(root, f)
            break
    if KB_PATH:
        break

if KB_PATH is None:
    raise FileNotFoundError("knowledge_base*.json not found in /kaggle/input. Attach dataset via Data tab.")

print(f"Loading KB from: {KB_PATH}")
with open(KB_PATH, encoding="utf-8") as f:
    kb_docs = json.load(f)

bm25_index, bm25_docs = build_bm25_index(kb_docs)
print(f"✅ BM25 pipeline ready — {len(kb_docs)} docs indexed")

In [ ]:
REAL_WORLD_CASES = [
  {"id": "rw_01", "category": "Race Condition",
   "code": "fs.exists(filePath, function(exists) {\n  if (exists) {\n    fs.readFile(filePath, 'utf8', callback);\n  }\n});",
   "pass_keywords": ["readfile", "enoent", "atomic", "access", "toctou", "race"]},
  {"id": "rw_02", "category": "Double Callback",
   "code": "function query(sql, cb) {\n  connection.connect(function(err) {\n    if (err) cb(err);\n    connection.query(sql, function(err, results) {\n      cb(err, results);\n    });\n  });\n}",
   "pass_keywords": ["return cb", "return callback", "double", "called twice", "early return"]},
  {"id": "rw_03", "category": "Zalgo",
   "code": "function getData(key, callback) {\n  if (cache[key]) {\n    callback(null, cache[key]);\n  } else {\n    fetchFromDB(key, function(err, data) {\n      cache[key] = data;\n      callback(err, data);\n    });\n  }\n}",
   "pass_keywords": ["nexttick", "process.nexttick", "setimmediate", "zalgo", "async", "consistent"]},
  {"id": "rw_04", "category": "Event Loop Blocking",
   "code": "app.use(function(req, res, next) {\n  const result = [];\n  for (let i = 0; i < req.body.items.length; i++) {\n    result.push(heavyTransform(req.body.items[i]));\n  }\n  res.json(result);\n});",
   "pass_keywords": ["worker", "setimmediate", "async", "chunk", "promise", "block"]},
  {"id": "rw_05", "category": "Context Loss",
   "code": "class UserController {\n  constructor() { this.db = new Database(); }\n  async getUser(req, res) {\n    const user = await this.db.find(req.params.id);\n    res.json(user);\n  }\n}\nconst ctrl = new UserController();\napp.get('/user/:id', ctrl.getUser);",
   "pass_keywords": ["bind", "arrow", "this", ".bind(ctrl)", "=> ctrl"]},
  {"id": "rw_06", "category": "Resource Exhaustion",
   "code": "pool.connect(function(err, client, done) {\n  client.query('BEGIN', function(err) {\n    if (err) {\n      client.query('ROLLBACK', function(err) { done(); callback(err); });\n    }\n    client.query(sql, function(err, result) {\n      if (err) {\n        client.query('ROLLBACK', function(err) { done(); });\n        callback(err);\n      }\n      client.query('COMMIT', function(err) { done(); callback(null, result); });\n    });\n  });\n});",
   "pass_keywords": ["done()", "release", "return", "finally", "leak"]},
  {"id": "rw_07", "category": "Stream Leak",
   "code": "function serveFile(req, res) {\n  const readStream = fs.createReadStream(req.params.file);\n  const gzip = zlib.createGzip();\n  readStream.pipe(gzip).pipe(res);\n}",
   "pass_keywords": ["pipeline", "error", "destroy", "on('error'", "cleanup"]},
  {"id": "rw_08", "category": "Race Condition",
   "code": "app.post('/login', function(req, res, next) {\n  passport.authenticate('local', function(err, user) {\n    if (err) return next(err);\n    req.logIn(user, function(err) {\n      if (err) return next(err);\n      return res.redirect('/');\n    });\n  })(req, res, next);\n});",
   "pass_keywords": ["session", "atomic", "race", "concurrent", "await"]},
  {"id": "rw_09", "category": "Unhandled Rejection",
   "code": "co(function*() {\n  const conn = yield db.connect();\n  const result = yield conn.query(sql);\n  return result;\n}).then(function(result) {\n  res.json(result);\n});",
   "pass_keywords": ["catch", "try", "rejection", ".catch(", "error"]},
  {"id": "rw_10", "category": "Resource Exhaustion",
   "code": "const readStream = fs.createReadStream('huge-file.csv');\nreadStream.on('data', function(chunk) {\n  const parsed = parseCSVChunk(chunk);\n  writableDB.write(parsed);\n});",
   "pass_keywords": ["pause", "resume", "pipe", "backpressure", "drain", "highwatermark"]},
  {"id": "rw_11", "category": "Double Callback",
   "code": "function getWithTimeout(key, timeout, cb) {\n  const timer = setTimeout(function() {\n    cb(new Error('timeout'));\n  }, timeout);\n  client.get(key, function(err, data) {\n    cb(err, data);\n  });\n}",
   "pass_keywords": ["cleartimeout", "clearTimeout", "called twice", "return cb", "once"]},
  {"id": "rw_12", "category": "Sequential Awaits",
   "code": "async function getDashboard(userId) {\n  const user = await db.users.findById(userId);\n  const orders = await db.orders.findByUser(userId);\n  const notifications = await db.notifications.findByUser(userId);\n  return { user, orders, notifications };\n}",
   "pass_keywords": ["promise.all", "parallel", "concurrent", "Promise.all"]},
  {"id": "rw_13", "category": "Race Condition",
   "code": "queue.process(function(job, done) {\n  if (job.data.status === 'pending') {\n    job.data.status = 'processing';\n    processJob(job.data, function(err, result) {\n      done(err, result);\n    });\n  }\n});",
   "pass_keywords": ["atomic", "race", "lock", "redis", "transaction", "update"]},
  {"id": "rw_14", "category": "Callback Hell",
   "code": "fs.readFile(configPath, function(err, config) {\n  if (err) return callback(err);\n  db.connect(JSON.parse(config), function(err, conn) {\n    if (err) return callback(err);\n    conn.query(sql, function(err, rows) {\n      if (err) return callback(err);\n      rows.forEach(function(row) {\n        transform(row, function(err, result) {\n          results.push(result);\n        });\n      });\n      callback(null, results);\n    });\n  });\n});",
   "pass_keywords": ["async/await", "async await", "promise", "flatten", "await"]},
  {"id": "rw_15", "category": "Resource Exhaustion",
   "code": "async function notifyAllUsers(users) {\n  await Promise.all(\n    users.map(user =>\n      axios.post('/notify', { userId: user.id })\n    )\n  );\n}",
   "pass_keywords": ["limit", "chunk", "batch", "p-limit", "concurren", "slice"]},
  {"id": "rw_16", "category": "Event Loop Blocking",
   "code": "app.post('/register', function(req, res) {\n  const hash = crypto.pbkdf2Sync(\n    req.body.password,\n    req.body.username,\n    100000, 64, 'sha512'\n  );\n  db.users.create({ hash }, function(err) {\n    res.json({ ok: true });\n  });\n});",
   "pass_keywords": ["pbkdf2", "async", "worker", "promise", "block"]},
  {"id": "rw_17", "category": "Zalgo",
   "code": "function processItems(items, callback) {\n  if (items.length === 0) { callback(null, []); return; }\n  const item = items[0];\n  if (computedCache[item]) {\n    processItems(items.slice(1), function(err, rest) {\n      callback(null, [computedCache[item]].concat(rest));\n    });\n  } else {\n    asyncCompute(item, function(err, result) {\n      computedCache[item] = result;\n      processItems(items.slice(1), function(err, rest) {\n        callback(null, [result].concat(rest));\n      });\n    });\n  }\n}",
   "pass_keywords": ["nexttick", "setimmediate", "async", "consistent", "zalgo"]},
  {"id": "rw_18", "category": "Unhandled Rejection",
   "code": "app.get('/user/:id', async function(req, res) {\n  const user = await db.findUser(req.params.id);\n  res.json(user);\n});",
   "pass_keywords": ["try", "catch", "next(err", "rejection", ".catch"]},
  {"id": "rw_19", "category": "Race Condition",
   "code": "function createFileIfNotExists(filePath, content, callback) {\n  fs.access(filePath, fs.constants.F_OK, function(err) {\n    if (err) {\n      fs.writeFile(filePath, content, callback);\n    } else {\n      callback(null);\n    }\n  });\n}",
   "pass_keywords": ["wx", "exclusive", "atomic", "flag", "race", "toctou"]},
  {"id": "rw_20", "category": "Context Loss",
   "code": "class DataPoller {\n  constructor(interval) {\n    this.data = [];\n    this.interval = interval;\n  }\n  start() {\n    setTimeout(function() {\n      this.data.push(Date.now());\n      setTimeout(arguments.callee, this.interval);\n    }, this.interval);\n  }\n}",
   "pass_keywords": ["arrow", "bind", "this", "=>", ".bind(this)"]},
  {"id": "rw_21", "category": "Sequential Awaits",
   "code": "app.post('/logout', (req, res) => {\n  req.logout(() => {\n    req.session.save();\n    res.redirect('/');\n  });\n});",
   "pass_keywords": ["await", "callback", "promise", "session.save", "then"]},
  {"id": "rw_22", "category": "Double Callback",
   "code": "async.series([\n  function(done) {\n    db.query('SELECT 1', function(err) {\n      done(err);\n      done(err);\n    });\n  },\n  function(done) {\n    done();\n  }\n], callback);",
   "pass_keywords": ["return done", "return callback", "once", "called twice", "guard"]},
  {"id": "rw_23", "category": "Resource Exhaustion",
   "code": "const txns = Array(50).fill(null).map(() =>\n  sequelize.transaction(t =>\n    User.create({ name: 'test' }, { transaction: t })\n  )\n);\nawait Promise.all(txns);",
   "pass_keywords": ["limit", "chunk", "batch", "pool", "concurren", "slice", "p-limit"]},
  {"id": "rw_24", "category": "Unhandled Rejection",
   "code": "const conn = mongoose.createConnection('mongodb://invalid-host:27017/db');\nconn.on('error', function(err) {\n  console.error('connection error:', err);\n});",
   "pass_keywords": ["catch", ".catch", "try", "rejection", "promise", "await"]},
  {"id": "rw_25", "category": "Race Condition",
   "code": "const subscriptions = ['ch1', 'ch2', 'ch3'];\nconst promises = subscriptions.map(ch =>\n  client.sUnsubscribe(ch)\n);\nawait Promise.all(promises);",
   "pass_keywords": ["sequential", "await", "race", "disconnect", "series", "loop"]},
  {"id": "rw_26", "category": "Event Loop Blocking",
   "code": "await Promise.all([\n  knex.transaction(t => t('users').forUpdate().select()),\n  knex.transaction(t => t('users').forUpdate().select()),\n  knex.transaction(t => t('users').forUpdate().select())\n]);",
   "pass_keywords": ["deadlock", "sequential", "lock", "timeout", "series", "queue"]},
  {"id": "rw_27", "category": "Sequential Awaits",
   "code": "app.use(function(req, res, next) {\n  req.session.touch();\n  req.session.userId = req.user.id;\n  next();\n});",
   "pass_keywords": ["await", "callback", "promise", "race", "async", "then"]},
  {"id": "rw_28", "category": "Unhandled Rejection",
   "code": "User.insertMany(\n  [{ name: 'Alice' }, { name: '' }],\n  function(err, docs) {\n    if (err) return console.log(err);\n    console.log(docs);\n  }\n);",
   "pass_keywords": ["catch", ".catch", "promise", "rejection", "try", "await"]},
  {"id": "rw_29", "category": "Double Callback",
   "code": "async function transfer(from, to, amount) {\n  const pipeline = client.pipeline();\n  pipeline.decrby(from, amount);\n  pipeline.incrby(to, amount);\n  await pipeline.exec();\n  await pipeline.exec();\n}",
   "pass_keywords": ["once", "return", "called twice", "remove", "exec once", "duplicate"]},
  {"id": "rw_30", "category": "Unhandled Rejection",
   "code": "app.use(async function(req, res, next) {\n  const data = await fetchData(req.params.id);\n  req.data = data;\n  next();\n});\n\napp.use(async function(req, res, next) {\n  const result = await processData(req.data);\n  res.json(result);\n});",
   "pass_keywords": ["try", "catch", "next(err", "rejection", "express-async-errors", ".catch"]}
]

In [ ]:
def score_response(keywords, response):
    if not response: return "fail"
    r = response.lower()
    matched = [kw for kw in keywords if kw.lower() in r]
    has_code = any(tok in r for tok in ["function", "const ", "async", "=>", "return", "await"])
    if len(matched) >= 2 and has_code: return "pass"
    elif len(matched) >= 1 or has_code: return "partial"
    return "fail"

In [ ]:
import time
MODEL_LABEL = "anha12/threadlearn-qwen2.5-coder-1.5b-merged-v2 + BM25+AST pipeline"
TOTAL = len(REAL_WORLD_CASES)
print("=" * 70)
print(f"ThreadLearn v2 — Real-World Benchmark w/ Pipeline ({TOTAL} cases)")
print(f"Model: {MODEL_LABEL}")
print("=" * 70)

results = []
pass_count = partial_count = fail_count = 0

for tc in REAL_WORLD_CASES:
    print(f"\n[{tc['id']}] {tc['category']}")
    t0 = time.time()
    query = extract_keywords(tc["code"])
    docs = bm25_search(bm25_index, bm25_docs, query, top_k=3)
    prompt = build_prompt(tc["code"], docs)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=300, do_sample=False, temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    latency = time.time() - t0
    verdict = score_response(tc["pass_keywords"], response)
    if verdict == "pass":      pass_count += 1;    icon = "✅ PASS   "
    elif verdict == "partial": partial_count += 1; icon = "⚠️  PARTIAL"
    else:                      fail_count += 1;    icon = "❌ FAIL   "
    print(f"  {icon} | {latency:.1f}s | BM25: {query[:50]}")
    print(f"  Docs: {[(d.get('title','')[:40], d.get('bm25_score')) for d in docs]}")
    print(f"  {response[:150].replace(chr(10), ' ')}...")
    results.append({"id": tc["id"], "category": tc["category"],
                    "verdict": verdict, "latency_s": round(latency, 2),
                    "bm25_query": query,
                    "docs_retrieved": [(d.get("title",""), d.get("bm25_score")) for d in docs],
                    "response_preview": response[:400]})

score = pass_count + partial_count * 0.5
print("\n" + "=" * 70)
print(f"RESULTS — {MODEL_LABEL}")
print(f"  Pass:    {pass_count}/{TOTAL}")
print(f"  Partial: {partial_count}/{TOTAL}")
print(f"  Fail:    {fail_count}/{TOTAL}")
print(f"  Score:   {score:.1f}/{TOTAL}  ({score/TOTAL*100:.1f}%)")
print("=" * 70)
output = {"model": MODEL_LABEL, "pipeline": "BM25+AST",
           "pass": pass_count, "partial": partial_count, "fail": fail_count,
           "score": f"{score:.1f}/{TOTAL}", "results": results}
with open("/kaggle/working/eval_merged_v2_pipeline_results.json", "w") as f:
    json.dump(output, f, indent=2)
print(f"\n💾 Saved: /kaggle/working/eval_merged_v2_pipeline_results.json")